# 创建索引（Index）

## 用途
教学演示 - 如何在 Milvus 中创建索引加速检索

## 核心概念
- 索引的作用和原理
- 常见索引类型对比（FLAT、IVF_FLAT、IVF_PQ、HNSW）
- 索引参数调优

## 运行前检查
1. 已安装依赖：`pip install -r requirements.txt`
2. 已完成 01-03 的学习（连接、建表、插入）
3. 确保 Collection 中有数据（运行本脚本会自动插入测试数据）

In [1]:
# 设置 UTF-8 编码（Windows 专用）

# 导入 Milvus 配置（优先使用 Docker Milvus）
import sys
import os
from rag_examples.milvus_config import MILVUS_URI

## 第一部分：理解索引

### 什么是索引（Index）？

**定义**
- 索引 = 加速检索的数据结构
- 类似于书籍的"目录"，让你快速找到目标内容

### 生活化比喻

- 没有索引 = 从头到尾翻书找内容（全表扫描）
- 有索引 = 查目录后直接翻到对应页（索引检索）

**例子：**
- 图书馆没有目录卡 → 要找一本书得走遍所有书架
- 图书馆有目录卡 → 先查卡片知道位置，直接去拿

### 索引加速原理
- **FLAT 索引**：不建索引，暴力搜索（适合小数据）
- **IVF 索引**：将向量分组，只在相关组内搜索
- **PQ 索引**：向量压缩，用更少的存储空间
- **HNSW 索引**：图结构，导航式搜索

### 索引类型对比

| 索引类型 | 适用场景 | 检索速度 | 精度 | 内存占用 |
|----------|----------|----------|------|----------|
| FLAT | <1 万条 | 慢 | 100% | 低 |
| IVF_FLAT | 1 万 -100 万 | 快 | 高 | 中 |
| IVF_PQ | >100 万 | 很快 | 中 | 低（压缩） |
| HNSW | 高精度场景 | 最快 | 非常高 | 高 |

### 注意事项
1. 数据量 < 1 万：用 FLAT 即可（简单直接）
2. 数据量 1 万 -100 万：用 IVF_FLAT（平衡速度和精度）
3. 数据量 > 100 万：考虑 IVF_PQ 或 HNSW
4. nlist 参数：IVF 的分组数，建议 √N（N 为向量数）

## 示例 1: 准备测试数据

In [2]:
def prepare_test_data():
    """
    准备测试用的文档数据
    """
    documents = [
        {"content": "人工智能是模拟人类智能的计算机科学领域。", "category": "AI"},
        {"content": "机器学习通过训练数据让计算机自动学习规律。", "category": "AI"},
        {"content": "深度学习使用多层神经网络模拟人脑。", "category": "AI"},
        {"content": "RAG 结合检索和生成技术，先检索再生成答案。", "category": "LLM"},
        {"content": "Milvus 是开源的向量数据库，支持亿级向量检索。", "category": "Database"},
        {"content": "自然语言处理让计算机理解和生成人类语言。", "category": "AI"},
        {"content": "计算机视觉让计算机能够'看懂'图像和视频。", "category": "AI"},
        {"content": "大语言模型是基于海量文本训练的深度学习和模型。", "category": "LLM"},
        {"content": "知识图谱用图结构存储和表示知识。", "category": "Knowledge"},
        {"content": "推荐系统根据用户偏好推荐相关内容。", "category": "Application"},
    ]

    return documents

In [3]:
def generate_mock_embeddings(texts, dim=768):
    """
    模拟 Embedding 生成的向量数据
    """
    import random
    random.seed(42)

    vectors = []
    for text in texts:
        vector = [random.random() for _ in range(dim)]
        vectors.append(vector)

    return vectors

In [4]:
def insert_test_data(client, collection_name):
    """
    向 Collection 插入测试数据
    """
    documents = prepare_test_data()
    texts = [d["content"] for d in documents]
    vectors = generate_mock_embeddings(texts)

    # 构造插入数据
    data_to_insert = []
    for i, doc in enumerate(documents):
        data_to_insert.append({
            "content": doc["content"],
            "category": doc["category"],
            "vector": vectors[i]  # 使用 vector 而不是 embedding
        })

    # 批量插入
    client.insert(collection_name, data_to_insert)

    # 重要：flush 使数据对搜索可见
    client.flush(collection_name=collection_name)

    print(f"✓ 已插入 {len(documents)} 条测试数据")

In [5]:
# 测试准备函数
docs = prepare_test_data()
print(f"准备了 {len(docs)} 条测试文档")

准备了 10 条测试文档


## 示例 2: FLAT 索引（精确搜索）

**FLAT = 不建索引，暴力搜索**

适合数据量小（<1 万条）、要求 100% 精度的场景。

In [15]:
def create_flat_index():
    """
    创建 FLAT 索引（精确搜索）

    FLAT = 不建索引，暴力搜索
    适合数据量小（<1 万条）、要求 100% 精度的场景
    """
    print("=" * 60)
    print("示例 2: FLAT 索引（精确搜索）")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "flat_index_demo"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 创建 Collection
    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=True,
        metric_type="COSINE"
    )

    # 插入测试数据
    insert_test_data(client, collection_name)

    # 创建 FLAT 索引
    # FLAT 索引实际上是不建索引，直接暴力搜索
    print("\n创建 FLAT 索引...")

    # 检查索引是否已存在
    index_list = client.list_indexes(collection_name=collection_name)
    client.release_collection(collection_name)
    if index_list:
        for index_name in index_list:
            client.drop_index(
                collection_name=collection_name,
                index_name=index_name
            )
        print(f"  已删除现有索引: {index_list}")

    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="vector",
        index_type="FLAT",  # 精确搜索
        metric_type="COSINE"
    )

    client.create_index(
        collection_name=collection_name,
        index_params=index_params
    )
    print("✓ FLAT 索引创建完成")

    # 查看索引信息
    index_info = client.describe_index(collection_name, "vector")
    print(f"  索引类型：{index_info['index_type']}")
    print(f"  度量类型：{index_info['metric_type']}")

    # 测试检索性能
    print("\n测试检索（FLAT 索引）：")
    test_vector = generate_mock_embeddings(["Milvus"])[0]

    # 加载collection
    client.load_collection(collection_name)

    res = client.search(
        collection_name=collection_name,
        data=[test_vector],
        limit=3,
        output_fields=["content"]
    )
    print(res)
    for i, hit in enumerate(res[0]):
        print(f"  {i+1}. 相似度：{hit['distance']:.4f} | 内容：{hit['entity']['content'][:20]}...")

    return client, collection_name

In [16]:
# 运行示例 2
client, collection_name = create_flat_index()

示例 2: FLAT 索引（精确搜索）
✓ 已插入 10 条测试数据

创建 FLAT 索引...
  已删除现有索引: ['vector']
✓ FLAT 索引创建完成
  索引类型：FLAT
  度量类型：COSINE

测试检索（FLAT 索引）：
data: [[{'id': 465448300126369397, 'distance': 0.9999998807907104, 'entity': {'content': '人工智能是模拟人类智能的计算机科学领域。'}}, {'id': 465448300126369403, 'distance': 0.7603273987770081, 'entity': {'content': "计算机视觉让计算机能够'看懂'图像和视频。"}}, {'id': 465448300126369398, 'distance': 0.7596487998962402, 'entity': {'content': '机器学习通过训练数据让计算机自动学习规律。'}}]]
  1. 相似度：1.0000 | 内容：人工智能是模拟人类智能的计算机科学领域。...
  2. 相似度：0.7603 | 内容：计算机视觉让计算机能够'看懂'图像和视频...
  3. 相似度：0.7596 | 内容：机器学习通过训练数据让计算机自动学习规律...


## 示例 3: IVF_FLAT 索引（倒排文件索引）

**IVF = Inverted File（倒排文件）**

将向量空间分成 nlist 个簇，只在相关簇内搜索。

In [ ]:
def create_ivf_flat_index():
    print("=" * 60)
    print("示例 3: IVF_FLAT 索引（倒排文件索引）")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "ivf_flat_index_demo"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 创建 Collection
    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=True,
        metric_type="COSINE"
    )

    # 插入测试数据
    insert_test_data(client, collection_name)

    # 创建 IVF_FLAT 索引
    print("\n创建 IVF_FLAT 索引...")

    # 释放集合（如果已加载）
    client.release_collection(collection_name=collection_name)

    # 删除现有索引
    index_list = client.list_indexes(collection_name=collection_name)
    if index_list:
        for index_name in index_list:
            client.drop_index(
                collection_name=collection_name,
                index_name=index_name
            )
        print(f"  已删除现有索引: {index_list}")

    # 创建 IVF_FLAT 索引
    nlist = 4
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="vector",
        index_type="IVF_FLAT",
        metric_type="COSINE",
        params={"nlist": nlist}
    )

    client.create_index(
        collection_name=collection_name,
        index_params=index_params
    )

    print(f"✓ IVF_FLAT 索引创建完成（nlist={nlist}）")

    # 加载集合
    client.load_collection(collection_name=collection_name)

    # 查看索引信息
    index_info = client.describe_index(collection_name, "vector")
    print(f"  索引类型：{index_info['index_type']}")
    print(f"  度量类型：{index_info['metric_type']}")

    # 测试检索
    print("\n测试检索（IVF_FLAT 索引）：")
    test_vector = generate_mock_embeddings(["测试查询"])[0]

    res = client.search(
        collection_name=collection_name,
        data=[test_vector],
        limit=3,
        output_fields=["content"],
        search_params={"params": {"nprobe": 3}}
    )

    for i, hit in enumerate(res[0]):
        print(f"  {i+1}. 相似度：{hit['distance']:.4f} | 内容：{hit['entity']['content'][:20]}...")

    return client, collection_name

In [ ]:
# 运行示例 3
client, collection_name = create_ivf_flat_index()

## 示例 4: HNSW 索引（图索引，高精度）

**HNSW = Hierarchical Navigable Small World**

通过构建多层图结构实现快速导航搜索。

In [ ]:
def create_hnsw_index():
    print("=" * 60)
    print("示例 4: HNSW 索引（图索引，高精度）")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "hnsw_index_demo"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 准备 HNSW 索引参数
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="vector",
        index_type="HNSW",
        metric_type="COSINE",
        params={
            "M": 16,
            "efConstruction": 200
        }
    )

    # 创建集合时指定索引
    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=True,
        metric_type="COSINE",
        index_params=index_params  # 直接指定索引
    )

    print("✓ 集合和 HNSW 索引创建完成")

    # 插入测试数据
    insert_test_data(client, collection_name)

    # 查看索引信息
    index_info = client.describe_index(collection_name, "vector")
    print(f"  索引类型：{index_info['index_type']}")
    print(f"  度量类型：{index_info['metric_type']}")

    # 测试检索
    print("\n测试检索（HNSW 索引）：")
    test_vector = generate_mock_embeddings(["测试查询"])[0]

    res = client.search(
        collection_name=collection_name,
        data=[test_vector],
        limit=3,
        output_fields=["content"],
        search_params={"params": {"ef": 64}}
    )

    for i, hit in enumerate(res[0]):
        print(f"  {i+1}. 相似度：{hit['distance']:.4f} | 内容：{hit['entity']['content'][:20]}...")

    return client, collection_name

In [ ]:
# 运行示例 4
client, collection_name = create_hnsw_index()

}## 示例 5: 索引类型详解

In [ ]:
def index_types_explained():
    """
    详解 Milvus 支持的索引类型
    """
    print("=" * 60)
    print("示例 5: 索引类型详解")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ Milvus 索引类型对比                                      │
├──────────────┬────────────────┬─────────────────────────┤
│    索引类型   │    适用场景     │         特点             │
├──────────────┼────────────────┼─────────────────────────┤
│ FLAT         │ <1 万条数据     │ 精确搜索，无需训练       │
│              │ 小数据量场景    │ 数据量小时最快           │
│              │                │ 100% 精度，无误差         │
├──────────────┼────────────────┼─────────────────────────┤
│ IVF_FLAT     │ 1 万 -100 万条   │ 倒排文件索引             │
│              │ 通用场景       │ 将向量分 nlist 组         │
│              │                │ 只在相关组内搜索         │
│              │                │ 需训练（build 阶段）      │
├──────────────┼────────────────┼─────────────────────────┤
│ IVF_PQ       │ >100 万条       │ 乘积量化索引             │
│              │ 超大数据量     │ 向量压缩存储             │
│              │                │ 内存占用低               │
│              │                │ 有一定精度损失           │
├──────────────┼────────────────┼─────────────────────────┤
│ IVF_SQ8      │ 内存受限场景    │ 标量量化索引             │
│              │                │ 8bit 压缩存储             │
│              │                │ 精度略低于 IVF_FLAT      │
├──────────────┼────────────────┼─────────────────────────┤
│ HNSW         │ 高精度场景      │ 可导航小世界图           │
│              │ 对延迟敏感     │ 多层图结构导航           │
│              │                │ 精度最高，速度最快       │
│              │                │ 内存占用较高             │
├──────────────┼────────────────┼─────────────────────────┤
│ ANNOY        │ 读多写少场景    │ 近似最近邻搜索           │
│              │                │ 构建快，查询快           │
│              │                │ 不支持动态增删           │
└──────────────┴────────────────┴─────────────────────────┘

💡 索引选择建议:

1. 数据量 < 1 万：
   → 选择 FLAT
   原因：数据量小，FLAT 足够快，且精度 100%

2. 数据量 1 万 - 100 万：
   → 选择 IVF_FLAT
   原因：平衡速度和精度，nlist = √N

3. 数据量 > 100 万：
   → 选择 IVF_PQ
   原因：压缩存储，节省内存

4. 高精度要求场景：
   → 选择 HNSW
   原因：精度最高，适合 RAG、推荐等场景

5. 内存受限场景：
   → 选择 IVF_SQ8
   原因：8bit 量化，内存占用低

📊 参数调优建议:

IVF_FLAT:
  - nlist: √N（N 为向量数）
  - nprobe: √nlist（搜索时探查的分组数）

HNSW:
  - M: 16-64（默认 16）
  - efConstruction: 200-400（构建时）
  - ef: 64-256（搜索时）

IVF_PQ:
  - m: 2-64（将向量分成 m 段）
  - nbits: 8（每段用 8bit 表示）
""")

In [ ]:
# 运行示例 5
index_types_explained()

## 示例 6: 索引管理操作

索引的查看、删除、重建等管理操作。

In [ ]:
def index_management():
    """
    索引的管理操作

    查看、删除、重建索引
    """
    print("=" * 60)
    print("示例 6: 索引管理操作")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "flat_index_demo"

    # 1. 查看索引
    print("1. 查看索引信息：")
    index_list = client.list_indexes(collection_name)
    if index_list:
        index_info = client.describe_index(collection_name, "vector")
        print(f"   索引类型：{index_info['index_type']}")
        print(f"   字段名：{index_info['field_name']}")
    else:
        print("   暂无索引")
    print()

    # 2. 删除索引
    print("2. 删除索引（示例代码）：")
    print("   client.drop_index(collection_name, 'embedding')")
    print()

    # 3. 重建索引
    print("3. 重建索引场景：")
    print("""
   何时需要重建索引？
   - 数据大量更新后
   - 索引参数需要调整
   - 索引损坏或异常

   重建步骤：
   1. 删除旧索引：client.drop_index(...)
   2. 创建新索引：client.create_index(...)
   3. 等待索引构建完成
""")

    # 4. 索引构建进度
    print("4. 查看索引构建进度（示例代码）：")
    print("""
   # 大量数据插入索引时，可以检查进度
   from pymilvus import Collection

   collection = Collection(collection_name)
   index_progress = client.get_index_build_progress(...)
   print(f"已扫描：{index_progress['total_rows']}")
   print(f"已索引：{index_progress['indexed_rows']}")
""")

In [ ]:
# 运行示例 6
index_management()

## 示例 7: 索引性能对比

In [ ]:
def index_performance_comparison():
    """
    索引性能对比说明
    """
    print("=" * 60)
    print("示例 7: 索引性能对比")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 不同索引类型性能对比（100 万向量，768 维）                 │
├──────────────┬─────────────┬────────────┬───────────────┤
│    索引类型   │  构建时间    │  检索延迟   │   召回率      │
├──────────────┼─────────────┼────────────┼───────────────┤
│ FLAT         │ 无需构建     │  ~500ms    │   100%       │
│ IVF_FLAT     │   ~30 秒     │  ~50ms     │   95-98%     │
│ IVF_PQ       │   ~20 秒     │  ~20ms     │   90-95%     │
│ HNSW         │   ~60 秒     │  ~10ms     │   98-99%     │
└──────────────┴─────────────┴────────────┴───────────────┘

💡 性能说明:

1. FLAT 索引
   - 优点：精度 100%，无需构建
   - 缺点：数据量大时极慢
   - 适用：数据量 < 1 万

2. IVF_FLAT 索引
   - 优点：速度快，精度高
   - 缺点：需要训练时间
   - 适用：通用场景，1 万 -100 万数据

3. IVF_PQ 索引
   - 优点：速度最快，内存省
   - 缺点：精度有损失
   - 适用：超大规模数据

4. HNSW 索引
   - 优点：速度最快，精度最高
   - 缺点：内存占用大
   - 适用：高精度、低延迟场景

📊 检索速度对比:

假设有 100 万向量:
- FLAT:    需要比较 100 万次 → 500ms
- IVF_FLAT: 只需比较 1 万 次 → 50ms (10 倍加速)
- HNSW:    图导航只需几步 → 10ms (50 倍加速)
""")

In [ ]:
# 运行示例 7
index_performance_comparison()